# 19. Correlation-Driven Framework for Spectrum Prediction

Implements the **correlation-driven** idea from the paper:
**"A Correlation-Driven Framework for Multivariate Time Series Forecasting"** (ICETM 2025).

## Paper (adapted to our setting)
- **Idea:** Transform time series into a **metric/correlation space** (pairwise correlation matrices) for prediction; reconstruction from predicted matrix can be done via closed-form (paper uses Pearson correlation, SVD-based reconstruction).
- **Adaptation:** We treat each 72h lookback as multivariate over **segments**: split 72 into N_SEG segments → N_SEG "series" of length segment_len. Build **Pearson correlation matrix** (N_SEG × N_SEG) between these segments. Use the flattened correlation matrix (upper triangle) as a **representation** of the input, then predict 24h from it (with optional raw features). This preserves the paper’s use of correlation as a structured representation for forecasting.

## Same setup as 10–18
Data: work_dir/final, 72h→24h. Metrics: MAE, RMSE, MASE. Models: Correlation-driven (corr features + MLP), Naive. Same visuals.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'):
            _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists():
        return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try:
            dfs.append(pd.read_parquet(p))
        except Exception as e:
            print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)

In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band:
        continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Correlation-driven representation (paper: pairwise correlation matrix)
Split lookback into N_SEG segments; compute Pearson correlation matrix (N_SEG × N_SEG); flatten upper triangle to get a fixed-size feature vector.

In [ ]:
N_SEG = 6
assert LOOKBACK % N_SEG == 0, "LOOKBACK must be divisible by N_SEG"
SEG_LEN = LOOKBACK // N_SEG

def build_correlation_features(x_72):
    """x_72: (72,) or (72, 1). Split into N_SEG segments, compute Pearson corr matrix, return upper triangle (incl diagonal)."""
    x = np.asarray(x_72).ravel()
    segs = np.array([x[i*SEG_LEN:(i+1)*SEG_LEN] for i in range(N_SEG)])
    if segs.shape[1] < 2:
        return np.zeros(N_SEG * (N_SEG + 1) // 2, dtype=np.float32)
    corr = np.corrcoef(segs)
    if not np.isfinite(corr).all():
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    triu_idx = np.triu_indices(N_SEG)
    return corr[triu_idx].astype(np.float32)

def build_correlation_dataset(X):
    """X: (B, 72, 1). Return (B, n_corr + 72) with corr features + raw for stability."""
    B = X.shape[0]
    n_corr = N_SEG * (N_SEG + 1) // 2
    out = np.zeros((B, n_corr + LOOKBACK), dtype=np.float32)
    for i in range(B):
        out[i, :n_corr] = build_correlation_features(X[i, :, 0])
        out[i, n_corr:] = X[i, :, 0].ravel()
    return out

n_corr = N_SEG * (N_SEG + 1) // 2
X_train_corr = build_correlation_dataset(X_train)
X_test_corr = build_correlation_dataset(X_test)
print(f"Correlation features: {n_corr} (upper triangle) + {LOOKBACK} raw = {X_train_corr.shape[1]} dims per sample.")

In [ ]:
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_train_corr_s = scaler_X.fit_transform(X_train_corr)
X_test_corr_s = scaler_X.transform(X_test_corr)
y_train_flat = y_train.reshape(-1, 1)
y_train_s = scaler_y.fit_transform(y_train_flat).reshape(y_train.shape)

BATCH = 128 if USE_GPU else 32
EPOCHS = 50
model_corr = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_corr.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(FORECAST_HORIZON, activation='linear'),
])
model_corr.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
model_corr.fit(X_train_corr_s, y_train_s, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
y_pred_corr_s = model_corr.predict(X_test_corr_s, verbose=0)
y_pred_corr = scaler_y.inverse_transform(y_pred_corr_s.reshape(-1, 1)).reshape(y_test.shape)
y_pred_corr = np.clip(y_pred_corr, 0, 100).astype(np.float32)
print('Correlation-driven model trained and predictions done.')

In [ ]:
def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_c = calculate_mae(y_test, y_pred_corr)
rmse_c = calculate_rmse(y_test, y_pred_corr)
mase_c = calculate_mase(y_test, y_pred_corr, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "Correlation-driven (MLP)", "MAE": mae_c, "RMSE": rmse_c, "MASE": mase_c},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY (same format as notebook 10)")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(f"\n{results_df.to_string(index=False)}")
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: Correlation-driven framework', y=1.02, fontsize=12)
plt.show()

In [ ]:
naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive Baseline (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_corr[i], '-', linewidth=1.6, label='Correlation-driven')
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted (solid) vs Actual (dashed)')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_corr.mean(axis=0), '-', linewidth=1.6, label='Correlation-driven (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mae_per_hour_c = np.abs(y_test - y_pred_corr).mean(axis=0)
mae_per_hour_naive = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour_c, '-o', label='Correlation-driven', markersize=4)
ax.plot(hours, mae_per_hour_naive, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals_best = (y_test - y_pred_corr).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_best, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: Correlation-driven')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive Baseline')
plt.suptitle('Residual distribution (centered at 0 is ideal)', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): Correlation-driven = {residuals_best.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         Correlation-driven = {residuals_best.std():.4f}, Naive = {residuals_naive.std():.4f}')

### Key insights

- **Paper:** Transforms time series into a metric/correlation space (pairwise Pearson correlation matrices); prediction and/or reconstruction in that space can improve accuracy. Full pipeline: predict next correlation matrix, then reconstruct next values via closed-form (SVD); with noise, use K matrices and minimum-diameter selection.
- **Adaptation:** We use the correlation matrix (over segments of the lookback) as a **feature representation**, then predict 24h directly with an MLP, avoiding non-convex reconstruction while keeping the correlation-driven representation.
- **Improvement over naive:** Positive % means the model beats the last-value baseline.
- **Mean profile / per-hour MAE / residuals:** Same interpretation as notebooks 10–18.

In [ ]:
best_row = results_df[results_df['Model'] == 'Correlation-driven (MLP)'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"Best model: Correlation-driven (MLP) (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}).")
print(f"Improvement over Naive: MAE {imp_mae:+.1f}%.")